#### LLM as Judges 
##### Lab 7.01 Healthcare: Patient Record Summarization
Dina Bosma-Buczynska

#### Part 2: Implementation & Execution
---

**Scenario:** A hospital wants an LLM to summarize patient records, preserving
all critical medical information. Key concerns: accuracy, completeness, terminology.

**Notebook follows the lab steps in order:**

| Lab step | What it does |
|---|---|
| Step 7 | Setup and environment |
| Step 8 | Implement the LLM-as-judge evaluation function |
| Step 9 | Create the test dataset |
| Step 10 | Run evaluation and collect metrics |
| Step 11 | Analyze and display results |



---
#### Step 7:Setup and Environment


##### Step 7a: Install required packages

Two packages are needed:
- `openai` - the official Python SDK for the OpenAI API
- `python-dotenv` - reads your `.env` file so the API key never appears in code

Run this cell once. You can skip it on future runs if already installed.

In [3]:
%pip install openai python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


##### Step 7b: Import libraries and initialize the client

**What each import does:**
- `os` - reads environment variables (where the API key lives after dotenv loads it)
- `json` - converts Python dicts to JSON text and back
- `time` - measures how long each evaluation call takes
- `OpenAI` - the client class; every API call goes through this object
- `load_dotenv` - reads the `.env` file and puts its contents into the environment

**Why `load_dotenv()` must come before `os.getenv()`:**

`load_dotenv()` reads the `.env` file and loads it into the process environment.
Only after that step does `os.getenv('OPENAI_API_KEY')` find anything.
If you swap the order, `os.getenv` returns `None` and the client creation fails.

In [4]:
import os
import json
import time
from openai import OpenAI
from dotenv import load_dotenv

# Step 1: load the .env file into the environment
load_dotenv()

# Step 2: read the key from the environment
api_key = os.getenv('OPENAI_API_KEY')

# Step 3: stop immediately with a clear message if the key is missing
if not api_key:
    raise ValueError(
        'OPENAI_API_KEY not found. '
        'Create a .env file with: OPENAI_API_KEY=sk-your-key-here'
    )

# Step 4: create the OpenAI client
# This is the object we call every time we want to send a request to the API
client = OpenAI(api_key=api_key)

# Use gpt-4o-mini - cost-efficient and free-tier friendly (as recommended by the lab)
MODEL = 'gpt-4o-mini'

print(f'Client ready. Model: {MODEL}')
print(f'API key loaded: {api_key[:8]}...{api_key[-4:]}')  # show only partial key

Client ready. Model: gpt-4o-mini
API key loaded: sk-svcac...fHkA


---
#### Step 8: Implement the Evaluation Prompt

The lab asks us to implement the LLM-as-judge prompt we designed in the written work (Step 4).

Create a function that takes the original prompt and the model's response,
and returns a score (1-5), reasoning, and a criteria checklist.

Also implement a helper for the rule-based keyword check: a fast, free safety
net that catches hard failures before the judge runs.

The judge prompt follows the 4-part structure from the lab:
1. Task description
2. Evaluation criteria
3. Reasoning steps
4. Output format (JSON)

##### Step 8a - Helper: generate_model_response()

Sends the clinical prompt to GPT-4o-mini and returns the summary plus token count.

**Key parameters:**
- `temperature=0.2` - slight variation, reflecting realistic deployment conditions.
  0 = always identical. 1 = very random. 0.2 is a realistic middle ground.
- `max_tokens=500` - caps response length. 500 tokens is roughly 375 words.
- `response.choices[0].message.content` - the API can return multiple responses
  at once. We always take the first one (`[0]`).
- `response.usage.total_tokens` - input tokens + output tokens combined.
  Tracked to estimate cost at the end.

In [5]:
def generate_model_response(prompt):
    """
    Send a clinical summarization prompt to the model and return its response.

    Parameters:
        prompt (str): The clinical record instruction.

    Returns:
        tuple: (response_text as str, tokens_used as int)
    """
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=500,
        temperature=0.2,  # slight variation to reflect realistic deployment
        messages=[
            {
                'role': 'system',
                'content': (
                    'You are a clinical documentation assistant. You summarize patient '
                    'records accurately and completely, always preserving critical medical '
                    'information including diagnoses, medications, and allergies.'
                )
            },
            {
                'role': 'user',
                'content': prompt
            }
        ]
    )

    # .choices[0] = first (and only) response
    # .message.content = the actual text the model returned
    response_text = response.choices[0].message.content

    # .usage.total_tokens = input + output tokens combined
    tokens_used = response.usage.total_tokens

    return response_text, tokens_used

print('generate_model_response() defined.')

generate_model_response() defined.


##### Step 8b: Helper: check_required_keywords()

The fast, free rule-based safety check. Runs before the LLM judge.

**Why have this if we already have an LLM judge?**

The judge costs tokens and takes seconds per call. A keyword check is instant
and free - it just asks: does this word appear in the text at all?
If a drug name or allergy is completely absent, we catch it immediately
without spending any API credits.

**How `.lower()` works:**

`'COPD'.lower()` returns `'copd'`. Convert both the response and each keyword
to lowercase before comparing, so 'COPD', 'copd', and 'Copd' all match correctly.

In [6]:
def check_required_keywords(response_text, required_keywords):
    """
    Rule-based safety check: verify all required clinical terms appear in the output.

    Parameters:
        response_text (str):      The model's generated summary.
        required_keywords (list): Terms that must appear (drug names, allergy names).

    Returns:
        dict: {
            'passed': True if all keywords found, False if any missing,
            'missing_keywords': list of terms not found in the response
        }
    """
    # Convert response to lowercase once so all comparisons are case-insensitive
    response_lower = response_text.lower()

    missing = []
    for keyword in required_keywords:
        if keyword.lower() not in response_lower:
            missing.append(keyword)  # this keyword was not found

    return {
        'passed': len(missing) == 0,  # True only if nothing is missing
        'missing_keywords': missing
    }

print('check_required_keywords() defined.')

check_required_keywords() defined.


##### Step 8c: The LLM-as-judge function: run_judge()

This is the core of the lab. The judge is a second LLM call that evaluates
the quality of the model's response.

**The judge receives:**
- The original prompt (what the model was asked to do)
- The model's response (what it actually produced)
- The evaluation criteria (what we care about for this test case)

**Three critical design choices explained:**

1. `response_format={'type': 'json_object'}` forces the model to return raw JSON.
   Without this, the model sometimes wraps its answer in markdown code fences
   like ```json ... ```, which breaks `json.loads()`. This flag prevents that.

2. `temperature=0` makes the judge as consistent as possible. We want the same
   input to produce the same score every time. The model being judged uses 0.2
   (realistic variation). The judge uses 0 (consistent, repeatable scoring).

3. `try/except json.JSONDecodeError` handles rare parse failures gracefully.
   Rather than crashing the whole script, we return a structured error result
   and flag the case for manual review.

**The judge prompt uses the 4-part structure from the lab (Step 4 of written work):**
Task description -> Evaluation criteria -> Reasoning steps -> Output format

In [7]:
def run_judge(original_prompt, model_response, expected_criteria):
    """
    Send the model response to a second LLM call (the judge) for structured scoring.

    Parameters:
        original_prompt (str):    The prompt originally sent to the model.
        model_response (str):     The summary the model produced.
        expected_criteria (dict): Criteria the judge should evaluate against.

    Returns:
        dict: Parsed JSON with score, reasoning, criteria_met, critical_flag.
    """

    # Build a readable criteria list for the judge prompt
    # e.g. '  - diagnostic_accuracy: COPD is named correctly'
    criteria_text = chr(10).join(
        f'  - {k}: {v}' for k, v in expected_criteria.items()
    )

    # The judge prompt - 4-part structure from the lab:
    # 1. Task description  2. Evaluation criteria  3. Reasoning steps  4. Output format
    judge_prompt = (
        'Task Description:\n'
        'A model was asked to summarize a clinical patient record for handover use.\n'
        'It must preserve all critical information: diagnoses, medications, allergies,\n'
        'and pending clinical actions.\n\n'
        'The original prompt given to the model:\n'
        '---\n'
        f'{original_prompt}\n'
        '---\n\n'
        'The model responded with:\n'
        '---\n'
        f'{model_response}\n'
        '---\n\n'
        'Evaluation Criteria:\n'
        f'{criteria_text}\n\n'
        'Reasoning Steps:\n'
        'Step 1: Check whether the primary diagnosis is correctly named.\n'
        '        Flag diagnostic_accuracy as false if absent or vaguely paraphrased.\n'
        'Step 2: Check whether all medications appear in the response.\n'
        '        Even one missing medication is a partial failure.\n'
        'Step 3: Check whether all allergies appear in the response.\n'
        '        A missing allergy is a patient safety issue - set critical_flag to true.\n'
        'Step 4: Assign a score 1-5. If allergy_safety is false, max score is 2.\n\n'
        'Score anchors:\n'
        '5 = All criteria met, clinically safe and complete\n'
        '4 = All criteria met, minor detail omission (e.g. one dosage missing)\n'
        '3 = Diagnosis and most medications correct, one allergy missing\n'
        '2 = Diagnosis correct, significant medication or allergy gaps\n'
        '1 = Multiple critical fields missing, not safe for clinical use\n\n'
        'Output Format:\n'
        'Return ONLY valid JSON. No preamble, no markdown, no text outside the object.\n'
        '{\n'
        '  "score": <integer 1-5>,\n'
        '  "reasoning": "<explanation referencing evidence from the model response>",\n'
        '  "criteria_met": {\n'
        '    "diagnostic_accuracy": <true or false>,\n'
        '    "medication_completeness": <true or false>,\n'
        '    "allergy_safety": <true or false>\n'
        '  },\n'
        '  "critical_flag": <true or false>,\n'
        '  "critical_flag_reason": "<safety issue if critical_flag is true, else null>"\n'
        '}'
    )

    # Send the judge prompt to the model
    # temperature=0 for maximum consistency
    # response_format forces raw JSON with no markdown wrapping
    response = client.chat.completions.create(
        model=MODEL,
        max_tokens=1000,
        temperature=0,
        response_format={'type': 'json_object'},
        messages=[
            {
                'role': 'system',
                'content': (
                    'You are a clinical AI evaluation expert. You evaluate whether LLM '
                    'responses to clinical summarization tasks are accurate, complete, '
                    'and safe. Always respond with valid JSON only.'
                )
            },
            {
                'role': 'user',
                'content': judge_prompt
            }
        ]
    )

    # Extract the raw text and parse as JSON
    raw_text = response.choices[0].message.content

    try:
        result = json.loads(raw_text)
    except json.JSONDecodeError:
        # Parsing failed: return a structured error and flag for manual review
        result = {
            'score': 0,
            'reasoning': f'JSON parse error. Raw: {raw_text[:200]}',
            'criteria_met': {
                'diagnostic_accuracy': False,
                'medication_completeness': False,
                'allergy_safety': False
            },
            'critical_flag': True,
            'critical_flag_reason': 'JSON parsing failed - manual review required'
        }

    # Add token count for cost tracking in Step 10
    result['tokens_used'] = response.usage.total_tokens

    return result

print('run_judge() defined.')

run_judge() defined.


---
#### Step 9: Create Test Dataset

The lab asks us to create a small dataset of 3-5 test cases based on the 5 evaluation
prompts designed in the written work (Step 3). Each test case is a Python dict containing:

| Field | What it is |
|---|---|
| `id` | Short label (TC1-TC5) for reference in results |
| `title` | Human-readable name for this test |
| `prompt` | The actual instruction sent to the model |
| `required_keywords` | Terms for the rule-based keyword check (Step 8b) |
| `expected_criteria` | What the LLM judge evaluates against (Step 8c) |

Each test case targets a different clinical failure mode:
- TC1: Are all medications preserved with dosages?
- TC2: Do allergies survive compression to 2 sentences?
- TC3: Is specialist terminology kept or simplified away?
- TC4: Are confidentiality constraints respected in a psychiatric note?
- TC5: Can the model handle 6 medications without dropping any?

In [8]:
test_cases = [
    {
        'id': 'TC1',
        'title': 'Medication Preservation',
        'prompt': (
            'You are a clinical documentation assistant. Summarize the following patient '
            'record in 3-5 sentences. Include the primary diagnosis, all medications, '
            'and any known allergies.\n\n'
            'Patient Record:\n'
            'Patient: Male, 67 years old.\n'
            'Admitted for: Acute exacerbation of COPD.\n'
            'Current medications: Tiotropium 18mcg inhaled daily, Salbutamol 100mcg PRN, '
            'Prednisolone 30mg oral for 5 days.\n'
            'Known allergies: Penicillin (anaphylaxis), NSAIDs (bronchospasm).\n'
            'Recent labs: SpO2 82% on room air, FEV1 45% predicted.'
        ),
        'required_keywords': ['COPD', 'Tiotropium', 'Salbutamol', 'Prednisolone',
                               'Penicillin', 'NSAID'],
        'expected_criteria': {
            'diagnostic_accuracy': 'COPD is named correctly',
            'medication_completeness': 'All three medications present',
            'allergy_safety': 'Both Penicillin and NSAIDs listed'
        }
    },
    {
        'id': 'TC2',
        'title': 'Allergy Retention Under Length Constraint',
        'prompt': (
            'Summarize the following patient record in exactly 2 sentences.\n\n'
            'Patient Record:\n'
            'Patient: Female, 43 years old.\n'
            'Admitted for: Appendectomy (laparoscopic).\n'
            'Medications: Metformin 500mg BD for Type 2 Diabetes.\n'
            'Allergies: Cephalosporins (rash), Latex (anaphylaxis).\n'
            'Post-op plan: Prophylactic antibiotics required. '
            'Discharge within 24 hours if stable.'
        ),
        'required_keywords': ['appendect', 'Metformin', 'Cephalosporin', 'Latex'],
        'expected_criteria': {
            'diagnostic_accuracy': 'Appendectomy identified as the procedure',
            'medication_completeness': 'Metformin present',
            'allergy_safety': 'Both Cephalosporins and Latex listed despite length constraint'
        }
    },
    {
        'id': 'TC3',
        'title': 'Diagnostic Accuracy With Specialist Terminology',
        'prompt': (
            'Summarize the following patient record for a general practitioner taking '
            'over ongoing care. Write 3-4 sentences.\n\n'
            'Patient Record:\n'
            'Patient: Female, 58 years old.\n'
            'Diagnosis: Stage IIIA non-small cell lung cancer (adenocarcinoma subtype, '
            'EGFR mutation positive).\n'
            'Treatment: Osimertinib 80mg daily (targeted therapy). '
            'Enrolled in palliative care review.\n'
            'Performance status: ECOG 2.\n'
            'Recent imaging: Mediastinal lymph node involvement confirmed.'
        ),
        'required_keywords': ['EGFR', 'Osimertinib', 'Stage IIIA', 'adenocarcinoma',
                               'palliative'],
        'expected_criteria': {
            'diagnostic_accuracy': 'Stage IIIA adenocarcinoma with EGFR status named',
            'medication_completeness': 'Osimertinib named (not paraphrased)',
            'allergy_safety': 'N/A - no allergies in this record'
        }
    },
    {
        'id': 'TC4',
        'title': 'Tone and Confidentiality in Psychiatric Handover',
        'prompt': (
            'Write a brief clinical handover summary for the incoming night shift nurse.\n\n'
            'Patient Record:\n'
            'Patient: Male, 29 years old.\n'
            'Admitted for: First-episode psychosis, suspected schizophrenia.\n'
            'Current medications: Olanzapine 10mg nocte (initiated today).\n'
            'Observations: Patient agitated earlier today, now calm. No current risk indicators.\n'
            'Family contact: Mother notified. Patient has not consented to further family disclosure.'
        ),
        'required_keywords': ['Olanzapine', 'psychosis', 'consent'],
        'expected_criteria': {
            'diagnostic_accuracy': 'Psychosis and suspected schizophrenia mentioned',
            'medication_completeness': 'Olanzapine 10mg present',
            'allergy_safety': 'Consent restriction on family disclosure respected'
        }
    },
    {
        'id': 'TC5',
        'title': 'Completeness With Complex Polypharmacy Record',
        'prompt': (
            'Summarize the following patient record in a structured format with these '
            'sections: Primary Diagnosis, Active Medications, Allergies, Pending Actions.\n\n'
            'Patient Record:\n'
            'Patient: Female, 74 years old.\n'
            'Primary diagnosis: Heart failure with reduced ejection fraction (HFrEF), EF 30%.\n'
            'Secondary diagnoses: Type 2 diabetes, chronic kidney disease Stage 3b, '
            'atrial fibrillation.\n'
            'Medications: Furosemide 40mg daily, Sacubitril/Valsartan 49/51mg BD, '
            'Bisoprolol 5mg daily, Apixaban 2.5mg BD, Metformin 500mg BD (under review '
            'given CKD), Atorvastatin 40mg nocte.\n'
            'Allergies: ACE inhibitors (dry cough), Sulfonamides (rash).\n'
            'Pending: Cardiology review in 3 days, nephrology referral awaiting, '
            'HbA1c due next week.'
        ),
        'required_keywords': ['HFrEF', 'Furosemide', 'Bisoprolol', 'Apixaban',
                               'Metformin', 'Atorvastatin', 'ACE', 'Sulfonamide',
                               'cardiology', 'nephrology'],
        'expected_criteria': {
            'diagnostic_accuracy': 'HFrEF with EF 30% identified as primary diagnosis',
            'medication_completeness': 'All 6 medications present',
            'allergy_safety': 'ACE inhibitors and Sulfonamides both listed'
        }
    }
]

print(f'Test dataset ready: {len(test_cases)} test cases')
for tc in test_cases:
    print(f"  {tc['id']}: {tc['title']}")

Test dataset ready: 5 test cases
  TC1: Medication Preservation
  TC2: Allergy Retention Under Length Constraint
  TC3: Diagnostic Accuracy With Specialist Terminology
  TC4: Tone and Confidentiality in Psychiatric Handover
  TC5: Completeness With Complex Polypharmacy Record


---
#### Step 10: Run Evaluation and Collect Metrics

The lab asks us to:
- Run all test cases through the judge function
- Collect metrics: scores, time, token usage, estimated cost
- Calculate aggregate statistics
- Save results to a JSON file

This step has three parts:
- **10a** - the orchestrator function that loops over all test cases
- **10b** - the single cell that triggers actual execution
- **10c** - saving results to `evaluation_results.json`

##### Step 10a - Orchestrator: run_full_evaluation()

This function ties together the three functions from Step 8.
For each test case it runs all three checks in sequence and records everything.

**`time.time()`** returns the current timestamp as a float (seconds since 1970).
Subtracting two `time.time()` calls gives elapsed seconds for that block of code.

**Cost estimate:** we use approximately $0.0003 per 1,000 tokens as a blended rate
for gpt-4o-mini. This is an approximation - actual billing splits input and output
tokens at different rates.

In [9]:
def run_full_evaluation(test_cases):
    """
    Orchestrate the full evaluation across all test cases.

    For each test case:
        1. Generate a model response (Step 8a)
        2. Run the rule-based keyword check (Step 8b)
        3. Run the LLM judge (Step 8c)
        4. Record timing and token metrics

    Parameters:
        test_cases (list): The test cases defined in Step 9.

    Returns:
        tuple: (list of individual result dicts, aggregate statistics dict)
    """
    all_results = []
    total_start = time.time()
    total_tokens = 0

    print(f'Model: {MODEL}')
    print(f'Running {len(test_cases)} test cases...')
    print('-' * 60)

    for tc in test_cases:
        print(f"\nEvaluating: {tc['id']} - {tc['title']}")
        case_start = time.time()

        # 1. Generate the model's summary for this prompt
        model_response, gen_tokens = generate_model_response(tc['prompt'])

        # 2. Fast rule-based check: are all required keywords present?
        keyword_check = check_required_keywords(model_response, tc['required_keywords'])

        # 3. LLM judge: structured scoring with reasoning
        judge_result = run_judge(
            original_prompt=tc['prompt'],
            model_response=model_response,
            expected_criteria=tc['expected_criteria']
        )

        # Record timing for this test case
        case_time = round(time.time() - case_start, 2)

        # Total tokens = generation tokens + judge tokens
        case_tokens = gen_tokens + judge_result.get('tokens_used', 0)
        total_tokens += case_tokens

        # Build the result record for this test case
        result = {
            'id': tc['id'],
            'title': tc['title'],
            'model_response': model_response,
            'keyword_check': keyword_check,
            'judge_score': judge_result.get('score'),
            'judge_reasoning': judge_result.get('reasoning'),
            'criteria_met': judge_result.get('criteria_met'),
            'critical_flag': judge_result.get('critical_flag'),
            'critical_flag_reason': judge_result.get('critical_flag_reason'),
            'time_seconds': case_time,
            'tokens_used': case_tokens
        }
        all_results.append(result)

        # Live status line printed as each case completes
        safety = 'CRITICAL FLAG' if result['critical_flag'] else 'OK'
        kw = 'PASS' if keyword_check['passed'] else 'FAIL'
        print(f"  Score: {result['judge_score']}/5 | Keywords: {kw} | Safety: {safety} | Time: {case_time}s")
        if not keyword_check['passed']:
            print(f"  Missing keywords: {keyword_check['missing_keywords']}")

    # Aggregate statistics
    total_time = round(time.time() - total_start, 2)
    scores = [r['judge_score'] for r in all_results if r['judge_score'] is not None]
    n = len(test_cases)

    # Blended cost estimate: ~$0.0003 per 1K tokens for gpt-4o-mini
    estimated_cost = round((total_tokens / 1000) * 0.0003, 4)

    aggregate = {
        'total_test_cases': n,
        'average_score': round(sum(scores) / len(scores), 2) if scores else 0,
        'min_score': min(scores) if scores else None,
        'max_score': max(scores) if scores else None,
        'critical_flags_count': sum(1 for r in all_results if r['critical_flag']),
        'keyword_check_failures_count': sum(1 for r in all_results if not r['keyword_check']['passed']),
        'total_time_seconds': total_time,
        'average_time_per_case_seconds': round(total_time / n, 2),
        'total_tokens_used': total_tokens,
        'estimated_cost_usd': estimated_cost
    }

    return all_results, aggregate

print('run_full_evaluation() defined.')

run_full_evaluation() defined.


##### Step 10b - Execute the evaluation

This is the cell that actually triggers all the API calls.
Every cell above was a definition only - nothing ran yet.

When you run this cell you will see a live status line for each test case.
The full run takes approximately 20-40 seconds.

In [10]:
# Run the full evaluation across all 5 test cases
results, aggregate = run_full_evaluation(test_cases)

print('\nDone.')

Model: gpt-4o-mini
Running 5 test cases...
------------------------------------------------------------

Evaluating: TC1 - Medication Preservation
  Score: 5/5 | Keywords: PASS | Safety: OK | Time: 7.05s

Evaluating: TC2 - Allergy Retention Under Length Constraint
  Score: 5/5 | Keywords: PASS | Safety: OK | Time: 4.87s

Evaluating: TC3 - Diagnostic Accuracy With Specialist Terminology
  Score: 5/5 | Keywords: PASS | Safety: OK | Time: 6.78s

Evaluating: TC4 - Tone and Confidentiality in Psychiatric Handover
  Score: 5/5 | Keywords: PASS | Safety: OK | Time: 6.04s

Evaluating: TC5 - Completeness With Complex Polypharmacy Record
  Score: 5/5 | Keywords: PASS | Safety: OK | Time: 5.64s

Done.


##### Step 10c - Save results to evaluation_results.json

The lab requires results saved in a structured JSON format containing both
individual results and aggregate statistics.

**`json.dump()` vs `json.dumps()`:**
- `json.dumps(data)` converts a Python dict to a JSON-formatted string
- `json.dump(data, file)` writes JSON directly to an open file object

`indent=2` adds 2-space indentation so the file is human-readable.
`ensure_ascii=False` lets medical symbols and special characters be stored as-is.

In [11]:
# Build the complete output structure
output = {
    'scenario': 'Healthcare - Patient Record Summarization',
    'model': MODEL,
    'aggregate': aggregate,
    'individual_results': results
}

# Save to JSON - this file is a required submission deliverable
output_path = 'evaluation_results.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f'Results saved to: {output_path}')

Results saved to: evaluation_results.json


---
##### Step 11: Analyze and Visualize Results

The lab asks us to display:
- Score distribution across all test cases
- Criteria performance (how many criteria were met per case)
- Time and cost metrics
- Key insights from the judge's reasoning

Three display cells below:
- **11a** - aggregate summary statistics
- **11b** - detailed per-case results with model response and judge reasoning
- **11c** - score distribution table

##### Step 11a - Aggregate statistics

In [12]:
print('=' * 60)
print('EVALUATION RESULTS - AGGREGATE SUMMARY')
print('=' * 60)
print(f"Scenario:               Healthcare / Patient Record Summarization")
print(f"Model:                  {MODEL}")
print(f"Total test cases:       {aggregate['total_test_cases']}")
print(f"Average judge score:    {aggregate['average_score']} / 5")
print(f"Score range:            {aggregate['min_score']} - {aggregate['max_score']}")
print(f"Critical safety flags:  {aggregate['critical_flags_count']}")
print(f"Keyword check failures: {aggregate['keyword_check_failures_count']}")
print(f"Total time:             {aggregate['total_time_seconds']}s")
print(f"Avg time per case:      {aggregate['average_time_per_case_seconds']}s")
print(f"Total tokens used:      {aggregate['total_tokens_used']}")
print(f"Estimated cost:         ${aggregate['estimated_cost_usd']} USD")

EVALUATION RESULTS - AGGREGATE SUMMARY
Scenario:               Healthcare / Patient Record Summarization
Model:                  gpt-4o-mini
Total test cases:       5
Average judge score:    5.0 / 5
Score range:            5 - 5
Critical safety flags:  0
Keyword check failures: 0
Total time:             30.38s
Avg time per case:      6.08s
Total tokens used:      5331
Estimated cost:         $0.0016 USD


##### Step 11b - Individual case results

Shows the full model response, judge reasoning, criteria checklist, and any
critical safety flags for each of the 5 test cases.

The judge reasoning is the most useful part to read - it explains exactly
what the model got right or wrong and why the score was assigned.

In [ ]:
print('=' * 60)
print('INDIVIDUAL RESULTS')
print('=' * 60)

for r in results:
    print(f"\n{'=' * 50}")
    print(f"{r['id']} | {r['title']}")
    print(f"{'=' * 50}")

    print(f"\nJUDGE SCORE:   {r['judge_score']} / 5")
    print(f"CRITICAL FLAG: {r['critical_flag']}")

    if r['critical_flag_reason']:
        print(f"FLAG REASON:   {r['critical_flag_reason']}")

    print(f"\nKEYWORD CHECK: {'PASSED' if r['keyword_check']['passed'] else 'FAILED'}")
    if r['keyword_check']['missing_keywords']:
        print(f"Missing terms: {r['keyword_check']['missing_keywords']}")

    print('\nCRITERIA MET:')
    if r['criteria_met']:
        for criterion, met in r['criteria_met'].items():
            print(f"  {criterion}: {'YES' if met else 'NO'}")

    print('\nMODEL RESPONSE:')
    print(r['model_response'])

    print('\nJUDGE REASONING:')
    print(r['judge_reasoning'])

    print(f"\nTime: {r['time_seconds']}s | Tokens: {r['tokens_used']}")

##### Step 11c - Score distribution table

A compact overview of all scores and critical flags side by side.
Any row marked CRITICAL needs manual review before this model is used in production.

In [13]:
print('SCORE DISTRIBUTION AND CRITERIA PERFORMANCE')
print('-' * 70)
print(f"{'Test Case':<38} {'Score':>5}  {'Criteria':>8}  {'Time':>6}  Flag")
print('-' * 70)

for r in results:
    flag = '*** CRITICAL' if r['critical_flag'] else 'OK'
    label = f"{r['id']} {r['title']}"
    criteria_count = sum(1 for v in r['criteria_met'].values() if v) if r['criteria_met'] else 0
    criteria_total = len(r['criteria_met']) if r['criteria_met'] else 0
    print(f"{label:<38} {str(r['judge_score']) + '/5':>5}  {str(criteria_count) + '/' + str(criteria_total):>8}  {str(r['time_seconds']) + 's':>6}  {flag}")

print('-' * 70)
print(f"{'AVERAGE':<38} {str(aggregate['average_score']) + '/5':>5}")

SCORE DISTRIBUTION AND CRITERIA PERFORMANCE
----------------------------------------------------------------------
Test Case                               Score  Criteria    Time  Flag
----------------------------------------------------------------------
TC1 Medication Preservation               5/5       3/3   7.05s  OK
TC2 Allergy Retention Under Length Constraint   5/5       3/3   4.87s  OK
TC3 Diagnostic Accuracy With Specialist Terminology   5/5       3/3   6.78s  OK
TC4 Tone and Confidentiality in Psychiatric Handover   5/5       3/3   6.04s  OK
TC5 Completeness With Complex Polypharmacy Record   5/5       3/3   5.64s  OK
----------------------------------------------------------------------
AVERAGE                                  5.0/5


##### Step 11d - Patterns and insights

Summarises criteria pass rates across all cases, truncated judge reasoning,
and identifies the key pattern from this evaluation run.

In [ ]:
criteria_keys = ['diagnostic_accuracy', 'medication_completeness', 'allergy_safety']

print('\n' + '=' * 60)
print('PATTERNS AND INSIGHTS')
print('=' * 60)

print('\nCriteria pass rate across all test cases:')
for key in criteria_keys:
    pass_count = sum(1 for r in results if r['criteria_met'] and r['criteria_met'].get(key))
    print(f'  {key}: {pass_count}/{len(results)}')

print('\nJudge reasoning summary (first 120 chars per case):')
for r in results:
    preview = r['judge_reasoning'][:120].rstrip() + '...' if len(r['judge_reasoning']) > 120 else r['judge_reasoning']
    print(f"  {r['id']}: {preview}")

print('\nKey finding:')
all_scores = [r['judge_score'] for r in results]
if all(s == 5 for s in all_scores):
    print(
        '  All cases scored 5/5 from the automated judge. This is consistent with\n'
        '  self-grading bias: the same model family was used for generation and\n'
        '  evaluation. See human_review_notes.md — TC2 and TC4 contain reasoning\n'
        '  failures the automated judge could not detect. Keyword presence is not\n'
        '  equivalent to clinical safety.'
    )
else:
    low = [r for r in results if r['judge_score'] < 4]
    print(f"  {len(low)} case(s) scored below 4/5: {', '.join(r['id'] for r in low)}")
    print('  Review critical_flag_reason fields for details.')


PATTERNS AND INSIGHTS

Criteria pass rate across all test cases:
  diagnostic_accuracy: 5/5
  medication_completeness: 5/5
  allergy_safety: 5/5

Judge reasoning summary (first 120 chars per case):
  TC1: The model accurately identified the primary diagnosis of COPD, included all current medications, and listed both known alle...
  TC2: The model accurately identified the diagnosis of laparoscopic appendectomy, included the medication Metformin 500mg BD for ...
  TC3: The model accurately summarized the patient's diagnosis as Stage IIIA non-small cell lung cancer (adenocarcinoma subtype, E...
  TC4: The model accurately summarized the patient's diagnosis of first-episode psychosis and suspected schizophrenia. It included...
  TC5: The model accurately identified the primary diagnosis of heart failure with reduced ejection fraction (HFrEF) and included ...

Key finding:
  All cases scored 5/5 from the automated judge. This is consistent with
  self-grading bias: the same model family w